In this notebook, I extend the modeling work from the previous notebooks to the day-ahead load forecasting problem. Previously, I forecasted load for a given hour using all available information up to that hour, including the actual temperature for the forecasted hour. However, such high-resolution data is not always available in practice. Day-ahead forecasting introduces additional constraints. First, recent load data are not available; the latest load observations are at least 24 hours old and may be even older, depending on the hour of prediction and the day-ahead market closing time. Second, actual temperature data for the prediction hour are unavailable, and models must rely on weather forecasts, which introduces an additional source of error.

To simulate these conditions, I enforce the following constraints on the test data. Only load lags greater than 24 hours are used as features, and actual temperature is replaced with forecasted temperature. Forecasted temperature is simulated by adding random noise to the actual temperature while incorporating autocorrelation with the forecasted temperature from the previous timestep. To reflect the increasing uncertainty of longer-horizon forecasts, the standard deviation of the noise is scaled linearly with the hour of day, reaching its maximum at hour 24.

In [427]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error

from itertools import product


In [1]:
def simulate_long_forecast(actual_temp_series, base_std = 1.0, max_std = 5.0, correlation_factor = 0.5):
    
    simulated_forecast = []
    last_forecast = actual_temp_series.iloc[0]
    
    hours_in_day = 24
    
    for i, actual in enumerate(actual_temp_series):
        hour_of_day = i % hours_in_day
        
        # Scale uncertainty linearly within the day
        hour_std = base_std + (max_std - base_std) * (hour_of_day / (hours_in_day - 1))
        
        # Random error for this hour
        error = np.random.normal(loc = 0, scale = hour_std)
        
        # Forecast combines correlation with last forecast and random error
        new_forecast = last_forecast + correlation_factor * (actual - last_forecast) + error
        
        simulated_forecast.append(new_forecast)
        last_forecast = new_forecast
    
    return pd.Series(simulated_forecast, index = actual_temp_series.index)

In [108]:
df = pd.read_csv("../data/processed/baseline_test.csv")[["timestamp", "Load", "avg_region_temp"]]
df = df[df["timestamp"] >= "2022-10-15 00:00:00"]

In [110]:
for i in range(10):
    simulated_temp = simulate_long_forecast(df['avg_region_temp']).rename(f"forecast_{i + 1}")
    df = pd.concat([df, simulated_temp], axis = 1)

In [112]:
base_t = 60

for i in range(10):
    df[f"temp_6h_{i + 1}"] = df[f"forecast_{i + 1}"].rolling(6).mean()
    df[f"CDH_{i + 1}"] = (df[f"forecast_{i + 1}"] - base_t).clip(lower=0)
    df[f"HDH_{i + 1}"] = (base_t - df[f"forecast_{i + 1}"]).clip(lower=0) 

df["Load_lag_24h"] = df["Load"].shift(24)
df["Load_lag_48h"] = df["Load"].shift(48)
df["temp_lag_24h"] = df["avg_region_temp"].shift(24)
df = df.dropna()

In [114]:
df

,timestamp,Load,avg_region_temp,forecast_1,forecast_2,forecast_3,forecast_4,forecast_5,forecast_6,forecast_7,...,HDH_8,temp_6h_9,CDH_9,HDH_9,temp_6h_10,CDH_10,HDH_10,Load_lag_24h,Load_lag_48h,temp_lag_24h
6936,2022-10-17 00:00:00,1944,61.880,60.924758,60.949440,61.539433,63.282522,63.723086,62.376279,64.165740,...,0.000000,66.266042,2.600669,0.000000,63.361962,0.181144,0.000000,1980.0,2010.0,63.644
6937,2022-10-17 01:00:00,1894,61.088,59.649839,58.439088,63.126342,63.401028,62.595623,58.876712,63.912191,...,0.000000,64.798605,1.242054,0.000000,62.704623,2.906308,0.000000,1918.0,1942.0,63.572
6938,2022-10-17 02:00:00,1900,60.836,59.930555,60.186675,60.399630,63.007095,60.099685,59.879146,62.649536,...,0.000000,63.755193,0.000000,0.097124,62.337672,1.001707,0.000000,1891.0,1915.0,63.572
6939,2022-10-17 03:00:00,1884,60.692,59.674823,61.433053,60.565345,61.476392,57.112419,62.841018,61.620100,...,0.000000,63.032325,2.785949,0.000000,62.298787,2.255890,0.000000,1860.0,1925.0,63.464
6940,2022-10-17 04:00:00,2004,60.908,59.482734,62.556699,59.564790,61.495258,57.138857,62.375521,61.958875,...,0.582610,62.133387,0.229818,0.000000,61.575272,1.544930,0.000000,1859.0,1929.0,63.932
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2022-12-31 19:00:00,2434,57.344,62.851656,58.621432,55.157078,55.745146,52.929522,56.782481,56.739527,...,2.291442,57.217667,0.000000,3.395091,57.757889,0.000000,2.418895,2505.0,2556.0,57.056
8756,2022-12-31 20:00:00,2343,57.596,60.838846,57.513774,57.802639,52.542405,54.997892,57.897436,54.793266,...,0.735037,56.915033,0.000000,0.900747,57.973273,0.000000,1.049775,2429.0,2469.0,57.020
8757,2022-12-31 21:00:00,2253,57.596,58.444235,56.836587,57.311022,54.694210,58.431454,56.744534,55.907006,...,7.531366,57.262317,0.000000,1.449986,57.255114,0.000000,6.123381,2332.0,2346.0,56.444
8758,2022-12-31 22:00:00,2159,57.272,61.784849,54.270597,55.554060,57.371098,55.317721,52.437990,58.841565,...,5.093096,57.480694,0.000000,3.210274,57.697226,0.000000,0.105777,2191.0,2189.0,56.516


In [248]:
df = pd.read_csv("../data/processed/day_ahead_train.csv")

In [309]:
df.columns

Index(['Year', 'Month', 'Day', 'Hour', 'Load', 'Site-1 Temp', 'Site-2 Temp',
       'Site-3 Temp', 'Site-4 Temp', 'Site-5 Temp', 'Site-1 GHI', 'Site-2 GHI',
       'Site-3 GHI', 'Site-4 GHI', 'Site-5 GHI', 'temp_actual',
       'temp_forecast_1', 'temp_forecast_2', 'temp_forecast_3',
       'temp_forecast_4', 'temp_forecast_5', 'temp_forecast_6',
       'temp_forecast_7', 'temp_forecast_8', 'temp_forecast_9',
       'temp_forecast_10', 'timestamp', 'Hour_sin', 'Hour_cos',
       'temp_6h_actual', 'CDH_actual', 'HDH_actual', 'temp_6h_forecast_1',
       'CDH_forecast_1', 'HDH_forecast_1', 'temp_6h_forecast_2',
       'CDH_forecast_2', 'HDH_forecast_2', 'temp_6h_forecast_3',
       'CDH_forecast_3', 'HDH_forecast_3', 'temp_6h_forecast_4',
       'CDH_forecast_4', 'HDH_forecast_4', 'temp_6h_forecast_5',
       'CDH_forecast_5', 'HDH_forecast_5', 'temp_6h_forecast_6',
       'CDH_forecast_6', 'HDH_forecast_6', 'temp_6h_forecast_7',
       'CDH_forecast_7', 'HDH_forecast_7', 'temp_6h_foreca

In [423]:
splits_df_loc = "../data/splits/split_bounds_day_ahead.csv"
splits_df = pd.read_csv(splits_df_loc)

def linear_rmse_on_features(df, features_to_train, do_print = False):

    if do_print:
        print("Now training on: ", features_to_train)

    rmse_list = []

    for i in range(1, 9):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask]
        val_split = df[val_mask]
    
        X_train = train_split[features_to_train]
        y_train = train_split["Load"]
    
        X_val = val_split[features_to_train]
        y_val = val_split["Load"]
    
        model = LinearRegression()
        model.fit(X_train, y_train)
    
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmse_list.append(rmse)
    if do_print:
        print("RMSEs: ", rmse_list)
        print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")
    return np.mean(rmse_list)

def linear_reg_rmse_on_features(df, features_to_train, regression_type = "Lasso", alpha = 0.1, do_print = False):

    if do_print:
        print("Now training on: ", features_to_train)

    rmse_list = []

    for i in range(1, 9):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask]
        val_split = df[val_mask]
    
        X_train = train_split[features_to_train]
        y_train = train_split["Load"]
    
        X_val = val_split[features_to_train]
        y_val = val_split["Load"]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        if regression_type == "Lasso":
            model = Lasso(alpha = alpha, max_iter=20_000)

        if regression_type == "Ridge":
            model = Ridge(alpha = alpha, max_iter=20_000)
            
        model.fit(X_train_scaled, y_train)
    
        y_pred = model.predict(X_val_scaled)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmse_list.append(rmse)
    if do_print:
        print("RMSEs: ", rmse_list)
        print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")
    return np.mean(rmse_list)


In [419]:
feat_list_one = ["temp_actual", "temp_6h_actual", "CDH_actual", "HDH_actual", "temp_actual_lag_24h", "Load_lag_24h", "Load_lag_48h", "is_weekend", "is_notable_day"]

max_hour_error = 0

for i in range(24):
    
    hour_df = df[df["Hour"] == i]
    max_hour_error = max(linear_rmse_on_features(hour_df, feat_list_one), max_hour_error)

print(max_hour_error)

alphas_lasso = np.logspace(-1, 1, 10)

for alpha in alphas_lasso:
    print(f"Lasso alpha = {alpha}")
    max_hour_error = 0
    
    for i in range(24):
        
        hour_df = df[df["Hour"] == i]
        max_hour_error = max(linear_reg_rmse_on_features(hour_df, feat_list_one, regression_type = "Lasso", alpha = alpha), max_hour_error)
    
    print(max_hour_error, "\n\n")

alphas_ridge = np.logspace(-3, 3, 13)

for alpha in alphas_ridge:
    print(f"Ridge alpha = {alpha}")
    max_hour_error = 0
    
    for i in range(24):
        
        hour_df = df[df["Hour"] == i]
        max_hour_error = max(linear_reg_rmse_on_features(hour_df, feat_list_one, regression_type = "Ridge", alpha = alpha), max_hour_error)
    
    print(max_hour_error, "\n\n")



253.06131510514382
Lasso alpha = 0.1
253.01632185242892 


Lasso alpha = 0.16681005372000587
252.97600821148524 


Lasso alpha = 0.2782559402207124
252.9126073498234 


Lasso alpha = 0.46415888336127786
252.81177032770776 


Lasso alpha = 0.774263682681127
252.65737722761992 


Lasso alpha = 1.291549665014884
252.43992471688665 


Lasso alpha = 2.1544346900318834
251.8241106439495 


Lasso alpha = 3.593813663804626
251.36608583814268 


Lasso alpha = 5.994842503189409
251.64960870395907 


Lasso alpha = 10.0
252.7839217427346 


Ridge alpha = 0.001
253.06110704099424 


Ridge alpha = 0.0031622776601683794
253.0606572723472 


Ridge alpha = 0.01
253.05923609166234 


Ridge alpha = 0.03162277660168379
253.05475299907926 


Ridge alpha = 0.1
253.040685380573 


Ridge alpha = 0.31622776601683794
252.99724409867247 


Ridge alpha = 1.0
252.8690706241931 


Ridge alpha = 3.1622776601683795
252.53127054156323 


Ridge alpha = 10.0
251.84549601687158 


Ridge alpha = 31.622776601683793
251.306

In [425]:
def xgbr_rmse_on_features(df, features_to_train, xgbr_depth = 3, xgbr_estimators = 200, xgbr_lr = 0.1, xgbr_min_child_weight = 5, do_print = False):

    if do_print:
        print("Now training on: ", features_to_train)
    
    rmse_list = []

    for i in range(1, 9):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask].drop(columns=["timestamp"])
        val_split = df[val_mask].drop(columns=["timestamp"])
    
        X_train = df[train_mask][features_to_train]
        y_train = df[train_mask]["Load"]
    
        X_val = df[val_mask][features_to_train]
        y_val = df[val_mask]["Load"]
    
        model = XGBRegressor(objective='reg:squarederror', n_estimators = xgbr_estimators, learning_rate = xgbr_lr, max_depth = xgbr_depth, min_child_weight = xgbr_min_child_weight, subsample=0.8)

        model.fit(X_train, y_train)
    
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmse_list.append(rmse)
        
    if do_print:
        print("RMSEs: ", rmse_list)
        print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")
    return np.mean(rmse_list)

In [465]:
depths = [i for i in range(2, 5)]
estimators = [i for i in range(50, 300, 50)]
min_child_weight = [10, 20]
lr = [0.05, 0.1]


param_grid = {
    "depth": depths,
    "estimators": estimators,
    "min_child_weight": min_child_weight,
    "lr": lr
}


results = []
for hour in range(0, 24):
    for depth, n_est, mcw, lr_ in product(
        param_grid["depth"],
        param_grid["estimators"],
        param_grid["min_child_weight"],
        param_grid["lr"]
    ):
        rmse = xgbr_rmse_on_features(
            df=df[df["Hour"] == hour],
            features_to_train=features_to_train,
            xgbr_depth=depth,
            xgbr_estimators=n_est,
            xgbr_lr=lr_,
            xgbr_min_child_weight=mcw,
            do_print=False
        )
    
        results.append({
            "hour": hour,
            "max_depth": depth,
            "n_estimators": n_est,
            "min_child_weight": mcw,
            "learning_rate": lr_,
            "mean_rmse": rmse
        })


In [467]:
results_df = pd.DataFrame(results)

rmse_by_hour = results_df.pivot_table(
    index=["max_depth", "n_estimators", "min_child_weight", "learning_rate"],
    columns="hour",
    values="mean_rmse"
).reset_index()

best_per_hour = results_df.loc[results_df.groupby("hour")["mean_rmse"].idxmin()].reset_index(drop=True)

In [469]:
best_per_hour

,hour,max_depth,n_estimators,min_child_weight,learning_rate,mean_rmse
0,0,3,100,10,0.10,83.053027
1,1,3,100,10,0.10,85.353300
2,2,3,100,10,0.10,78.098047
3,3,2,200,20,0.05,74.902996
4,4,4,100,10,0.05,79.101323
5,5,4,50,10,0.10,94.149697
6,6,4,100,10,0.05,119.035051
7,7,2,150,10,0.10,151.993966
8,8,3,150,10,0.10,172.167953
9,9,2,250,10,0.10,190.171149


In [323]:
df_test = pd.read_csv("../data/processed/day_ahead_test.csv")

In [ ]:
## 1 model for each hour, 24 models
